In [1]:
import awkward as ak
import numpy as np
import yaml

import matplotlib.pyplot as plt
import mplhep as hep

import hist
from hist import Hist

from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from topcoffea.modules.histEFT import HistEFT
NanoAODSchema.warn_missing_crossrefs = False

from coffea.analysis_tools import PackedSelection
from topcoffea.modules import utils
import topcoffea.modules.eft_helper as efth
from topcoffea.modules.paths import topcoffea_path

import re

In [33]:
import ttbarEFT.modules.object_selection as tt_os
import ttbarEFT.modules.event_selection as tt_es

import ttbarEFT.modules.corrections as tt_cor 
from topcoffea.modules.get_param_from_jsons import GetParam
get_tc_param = GetParam(topcoffea_path("params/params.json"))
from ttbarEFT.modules.paths import ttbarEFT_path

# import CorrectedJetsFactory
# import CorrectedMETFactory
from JECStack import JECStack
from ttbarEFT.modules.variation_filters import normalize_allowed_variations

In [3]:
clib_year_map = {
    "2016APV": "2016preVFP_UL",
    "2016preVFP": "2016preVFP_UL",
    "2016": "2016postVFP_UL",
    "2017": "2017_UL",
    "2018": "2018_UL",
    "2022": "2022_Summer22",
    "2022EE": "2022_Summer22EE",
    "2023": "2023_Summer23",
    "2023BPix": "2023_Summer23BPix",
}

In [6]:
fname = "/cms/cephfs/data/store/user/hnelson2/skims/mc/ptSkim/background/UL17_TTGJets/output_614.root"
events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "TTto2L2Nu"},
    mode="eager",
    entry_stop=10000,
).events()

In [7]:
year = "2017"
isData=False

In [8]:
######### Initialize Objects #########
met  = events.MET
ele  = events.Electron
mu   = events.Muon
tau  = events.Tau
jets = events.Jet 

leptonSelection = tt_os.Run2LeptonSelection()

# An array of length events that is just 1 for each event
events['nom'] = ak.ones_like(events.MET.pt)

######### Electron Selection ##########
ele['isGoodElec']=leptonSelection.is_sel_ele(ele)
ele_good = ele[ele.isGoodElec]

######### Muon Selection ##########
mu['pt'] = tt_cor.ApplyMuonPtCorr(mu, year, isData)
mu['isGoodMuon']=leptonSelection.is_sel_muon(mu)
mu_good = mu[mu.isGoodMuon]

In [9]:
jets['isClean'] = tt_os.isClean(jets, ele_good, drmin=0.4)& tt_os.isClean(jets, mu_good, drmin=0.4)
cleanedJets = jets[jets.isClean]
cleanedJets = ak.fill_none(cleanedJets, [])

# Medium DeepJet WP
medium_tag = "btag_wp_medium_" + year.replace("201", "UL1")
btagwpm = get_tc_param(medium_tag)

In [10]:
btag_eff_lookup_m = tt_cor.GetBtagEffLookup(year, wp='medium')
light_btag_SF_lookup = tt_cor.GetBtagSFLookup(wp='M',year=year, method='deepJet_incl')
bc_btag_SF_lookup = tt_cor.GetBtagSFLookup(wp='M',year=year, method='deepJet_comb')

In [11]:
cleanedJets["pt_raw"] = (1 - cleanedJets.rawFactor)*cleanedJets.pt
cleanedJets["mass_raw"] = (1 - cleanedJets.rawFactor)*cleanedJets.mass
rho_jagged = ak.ones_like(cleanedJets.pt) * events.fixedGridRhoFastjetAll
cleanedJets = ak.with_field(cleanedJets, rho_jagged, "Rho")
cleanedJets["pt_gen"] = ak.values_astype(ak.fill_none(cleanedJets.matched_gen.pt, 0), np.float32)

In [12]:
cleanedJets['pt'][0]

<Array [182, 75.4, 40, 33.1, 22, 19.3] type='6 * float32[parameters={"__doc...'>

In [13]:
# cleanedJets.matched_gen.pt

In [4]:
def get_jerc_keys(year, isdata, era=None):

    #JERC dictionary for various keys
    with open(ttbarEFT_path("modules/jerc_dict.yaml"), "r") as f:
        jerc_dict = yaml.safe_load(f)

    # Jet Algorithm
    if year.startswith("202"):
        jet_algo = 'AK4PFPuppi'
        jec_levels = jerc_dict[year]['jec_levels']
    else:
        jet_algo = 'AK4PFchs'
        jec_levels = []            # no JEC corrections for Run2, already applied in nanoAODv9
    

    # jerc keys and junc types
    if not isdata:
        jec_key    = jerc_dict[year]['jec_mc']
        jer_key    = jerc_dict[year]['jer']
        junc_types = jerc_dict[year]['junc']
    else:
        # if year in ['2016']: #,'2022','2023BPix'
        #     jec_key = jerc_dict[year]['jec_data']
        # else:
        #     jec_key = jerc_dict[year]['jec_data'][era]
        jec_key     = None
        jer_key     = None
        junc_types  = None

    return jet_algo, jec_key, jec_levels, jer_key, junc_types


def get_supported_jet_systematics(year, isData=False, era=None):
    if isData:
        return []

    systs = [f"JER_{year}Up", f"JER_{year}Down"]
    for base in get_supported_jes_bases(year, isData=isData, era=era):
        systs.append(f"JES_{base}Up")
        systs.append(f"JES_{base}Down")
    return systs



def get_supported_jes_bases(year, isData=False, era=None):
    if isData:
        return []

    jet_algo, jec_tag, _, _, junc_types = get_jerc_keys(year, isData, era)

    bases = []
    for junc_type in junc_types:
        full_unc_name = f"{jec_tag}_{junc_type}_{jet_algo}"
        bases.append(get_jec_uncertainty_label(full_unc_name, jec_tag, jet_algo))

    if len(set(bases)) != len(bases):
        duplicates = sorted({x for x in bases if bases.count(x) > 1})
        raise RuntimeError(
            f"Collision in JES base labels for year {year}: {duplicates}"
        )

    return bases


def get_jec_uncertainty_label(junc_name, jec_tag, jet_algo):
    """
    Extract the uncertainty label from a full correction name:
      {jec_tag}_{junc_type}_{jet_algo}
    """
    prefix = f"{jec_tag}_"
    suffix = f"_{jet_algo}"
    if not junc_name.startswith(prefix):
        raise ValueError(
            f'Uncertainty name "{junc_name}" does not start with expected prefix "{prefix}".'
        )
    if not junc_name.endswith(suffix):
        raise ValueError(
            f'Uncertainty name "{junc_name}" does not end with expected suffix "{suffix}".'
        )
    junc_type = junc_name[len(prefix) : -len(suffix)]
    if not junc_type:
        raise ValueError(f'Failed to extract junc_type from uncertainty name "{junc_name}".')
    return _canonicalize_junc_type_label(junc_type, jec_tag)


def _is_run2_jec_tag(jec_tag):
    return "UL" in jec_tag


def _canonicalize_junc_type_label(junc_type, jec_tag):
    # Preserve existing Run2 naming: regrouped UL sources drop the "Regrouped_" prefix.
    if _is_run2_jec_tag(jec_tag) and junc_type.startswith("Regrouped_"):
        return junc_type.replace("Regrouped_", "", 1)
    return junc_type

In [132]:
# import numpy
# import awkward as ak
import warnings
import logging
from functools import reduce

# from topcoffea.modules.JECStack import JECStack
# from JECStack import JECStack
# from topcoffea.modules.variation_filters import normalize_allowed_variations
# from ttbarEFT.modules.variation_filters import normalize_allowed_variations


_stack_parts = ["jec", "junc", "jer", "jersf"]
_MIN_JET_ENERGY = numpy.array(1e-2, dtype=numpy.float32)

logger = logging.getLogger(__name__)


def _random_gauss(counts, rng):
    total = int(ak.sum(counts))
    draws = rng.standard_normal(size=total).astype(numpy.float32)
    return ak.unflatten(draws, counts, axis=0)


def _as_jagged_per_jet(arr, counts, label):
    array = arr if isinstance(arr, ak.Array) else ak.Array(arr)
    layout = ak.to_layout(array)
    if getattr(layout, "is_list", False):
        return array

    expected = int(ak.sum(counts))
    if len(array) != expected:
        raise ValueError(f"{label} must return {expected} entries (one per jet); got {len(array)}")

    return ak.unflatten(array, counts, axis=0)


def _to_float32(values):
    if isinstance(values, ak.Array):
        return ak.values_astype(values, numpy.float32)
    return numpy.asarray(values, dtype=numpy.float32)


def jer_smear(
    variation,
    forceStochastic,
    pt_gen,
    jetPt,
    etaJet,
    jet_energy_resolution,
    jet_resolution_rand_gauss,
    jet_energy_resolution_scale_factor,
):
    if not isinstance(jetPt, ak.Array):
        raise Exception("'jetPt' must be an awkward array of some kind!")

    jetPt, pt_gen, etaJet, jet_energy_resolution, jet_resolution_rand_gauss = ak.broadcast_arrays(
        jetPt,
        ak.zeros_like(jetPt) if forceStochastic else pt_gen,
        etaJet,
        jet_energy_resolution,
        jet_resolution_rand_gauss,
    )

    jersf = ak.broadcast_arrays(jet_energy_resolution_scale_factor[..., variation], jetPt)[0]
    deltaPtRel = (jetPt - pt_gen) / jetPt
    doHybrid = (pt_gen > 0) & (numpy.abs(deltaPtRel) < 3 * jet_energy_resolution)
    detSmear = 1 + (jersf - 1) * deltaPtRel
    stochSmear = 1 + numpy.sqrt(numpy.maximum(jersf**2 - 1, 0)) * (jet_energy_resolution * jet_resolution_rand_gauss)

    min_jet_pt = _MIN_JET_ENERGY / numpy.cosh(etaJet)
    min_jet_pt_corr = min_jet_pt / jetPt
    smearfact = ak.where(doHybrid, detSmear, stochSmear)
    smearfact = ak.where((smearfact * jetPt) < min_jet_pt, min_jet_pt_corr, smearfact)
    return smearfact


def get_corr_inputs(jets, corr_obj, name_map, corrections=None):
    """Helper function for getting values of input variables."""

    input_values = []
    for inp in corr_obj.inputs:
        if inp.name == "systematic":
            continue
        if corrections is not None and inp.name == "JetPt":
            rawvar = ak.flatten(jets[name_map[inp.name]])
            correction = ak.flatten(corrections)
            input_values.append(correction * rawvar)
        else:
            input_values.append(ak.flatten(jets[name_map[inp.name]]))
    return input_values


class CorrectedJetsFactory(object):
    def __init__(self, name_map, jec_stack, allowed_variations=None):
        if not isinstance(jec_stack, JECStack):
            raise TypeError("jec_stack must be an instance of JECStack")

        self.tool = "clib" if jec_stack.use_clib else "jecstack"
        self.forceStochastic = False
        self._variation_control = normalize_allowed_variations(allowed_variations)

        if "ptRaw" not in name_map or name_map["ptRaw"] is None:
            warnings.warn(
                "There is no name mapping for ptRaw," " CorrectedJets will assume that <object>.pt is raw pt!"
            )
            name_map["ptRaw"] = name_map["JetPt"] + "_raw"
        self.treat_pt_as_raw = "ptRaw" not in name_map

        if "massRaw" not in name_map or name_map["massRaw"] is None:
            warnings.warn(
                "There is no name mapping for massRaw," " CorrectedJets will assume that <object>.mass is raw mass!"
            )
            name_map["massRaw"] = name_map["JetMass"] + "_raw"

        self.jec_stack = jec_stack
        self.name_map = name_map
        self.real_sig = [v for v in name_map.values()]

        if self.jec_stack.use_clib:
            self.load_corrections_clib()
        else:
            self.load_corrections_jecstack()

        if "ptGenJet" not in name_map:
            warnings.warn(
                'Input JaggedCandidateArray must have "ptGenJet" in order to apply hybrid JER smearing method. Stochastic smearing will be applied.'
            )
            self.forceStochastic = True

    def load_corrections_clib(self):
        self.corrections = self.jec_stack.corrections

    def load_corrections_jecstack(self):
        self.corrections = self.jec_stack.corrections

        total_signature = set()
        for part in _stack_parts:
            attr = getattr(self.jec_stack, part)
            if attr is not None:
                total_signature.update(attr.signature)

        missing = total_signature - set(self.name_map.keys())
        if len(missing) > 0:
            raise Exception(
                f"Missing mapping of {missing} in name_map!" + " Cannot evaluate jet corrections!" + " Please supply mappings for these variables!"
            )

    def build(self, jets):
        jets = ak.Array(jets)

        fields = ak.fields(jets)
        if len(fields) == 0:
            raise Exception("Empty record, please pass a jet object with at least {self.real_sig} defined!")

        counts = ak.num(jets, axis=1)
        base_parameters = ak.parameters(jets) or {}
        record_parameters = ak.parameters(ak.flatten(jets, axis=None)) or {}
        parameters = dict(record_parameters)
        parameters.update(base_parameters)
        parameters["corrected"] = True
        behavior = jets.behavior

        in_dict = {field: jets[field] for field in fields}
        out_dict = dict(in_dict)

        out_dict[self.name_map["JetPt"] + "_orig"] = out_dict[self.name_map["JetPt"]]
        out_dict[self.name_map["JetMass"] + "_orig"] = out_dict[self.name_map["JetMass"]]
        if self.treat_pt_as_raw:
            out_dict[self.name_map["ptRaw"]] = out_dict[self.name_map["JetPt"]]
            out_dict[self.name_map["massRaw"]] = out_dict[self.name_map["JetMass"]]

        jec_name_map = dict(self.name_map)
        jec_name_map["JetPt"] = jec_name_map["ptRaw"]
        jec_name_map["JetMass"] = jec_name_map["massRaw"]

        total_correction = None
        if self.tool == "jecstack":
            if self.jec_stack.jec is not None:
                jec_args = {k: ak.flatten(out_dict[jec_name_map[k]]) for k in self.jec_stack.jec.signature}
                corr = self.jec_stack.jec.getCorrection(**jec_args)
                total_correction = _as_jagged_per_jet(corr, counts, "JEC correction")
            else:
                total_correction = ak.ones_like(out_dict[self.name_map["JetPt"]])

        elif self.tool == "clib":
            total_correction = ak.values_astype(ak.ones_like(out_dict[self.name_map["JetPt"]]), numpy.float32)
            corrections_list = []
            for lvl in self.jec_stack.jec_names_clib:
                cumCorr = None
                if len(corrections_list) > 0:
                    cumCorr = ak.ones_like(total_correction)
                    for corr in corrections_list:
                        cumCorr = ak.values_astype(cumCorr * corr, numpy.float32)

                    # cmssw multiplies each successive correction by all previous corrections
                    # as part of the correction inputs
                sf = self.corrections.get(lvl, None)
                if sf is None:
                    raise ValueError(f"Correction {lvl} not found in self.corrections")

                inputs = get_corr_inputs(jets=jets, corr_obj=sf, name_map=jec_name_map, corrections=cumCorr)
                correction = _to_float32(sf.evaluate(*inputs))
                jagged_correction = _as_jagged_per_jet(correction, counts, f"Correction {lvl}")
                corrections_list.append(jagged_correction)
                total_correction = ak.values_astype(total_correction * jagged_correction, numpy.float32)

                if self.jec_stack.savecorr:
                    jec_lvl_tag = "_jec_" + lvl

                    out_dict[f"jet_energy_correction_{lvl}"] = jagged_correction
                    out_dict[self.name_map["JetPt"] + f"_{lvl}"] = jagged_correction * out_dict[self.name_map["ptRaw"]]
                    out_dict[self.name_map["JetMass"] + f"_{lvl}"] = jagged_correction * out_dict[self.name_map["massRaw"]]

                    out_dict[self.name_map["JetPt"] + jec_lvl_tag] = out_dict[self.name_map["JetPt"] + f"_{lvl}"]
                    out_dict[self.name_map["JetMass"] + jec_lvl_tag] = out_dict[self.name_map["JetMass"] + f"_{lvl}"]

            if total_correction is None:
                total_correction = ak.ones_like(out_dict[self.name_map["JetPt"]])

        out_dict["jet_energy_correction"] = total_correction

        out_dict[self.name_map["JetPt"]] = out_dict["jet_energy_correction"] * out_dict[self.name_map["ptRaw"]]
        out_dict[self.name_map["JetMass"]] = out_dict["jet_energy_correction"] * out_dict[self.name_map["massRaw"]]

        out_dict[self.name_map["JetPt"] + "_jec"] = out_dict[self.name_map["JetPt"]]
        out_dict[self.name_map["JetMass"] + "_jec"] = out_dict[self.name_map["JetMass"]]

        jagged_out = dict(out_dict)
        jer_systematic = None
        jes_systematics = {}

        jagged_out = dict(out_dict)
        jer_systematic = None
        jes_systematics = {}

        # --- JER availability check ---
        if self.tool == "jecstack":
            has_jer = self.jec_stack.jer is not None and self.jec_stack.jersf is not None
        elif self.tool == "clib":
            has_jer = len(self.jec_stack.jer_names_clib) > 0
        else:
            has_jer = False

        # Always apply nominal JER if available;
        # only build JER up/down systematics when variations are allowed.
        jer_nominal_enabled = has_jer
        jer_systematics_enabled = has_jer and self._variation_control.allow_jer

        if jer_nominal_enabled:
            jer_name_map = dict(self.name_map)
            jer_name_map["JetPt"] = jer_name_map["JetPt"] + "_jec"
            jer_name_map["JetMass"] = jer_name_map["JetMass"] + "_jec"

            if self.tool == "jecstack":
                jer_args = {k: ak.flatten(jagged_out[jer_name_map[k]]) for k in self.jec_stack.jer.signature}
                jer_resolution_flat = self.jec_stack.jer.getResolution(**jer_args)
                jet_energy_resolution = _as_jagged_per_jet(jer_resolution_flat, counts, "JER resolution")

                jersf_args_flat = {
                    k: ak.flatten(jagged_out[jer_name_map[k]], axis=None) for k in self.jec_stack.jersf.signature
                }
                try:
                    jersf_flat = self.jec_stack.jersf.getScaleFactor(**jersf_args_flat)
                except Exception as flat_exc:
                    # Some implementations expect jagged inputs and flatten internally.
                    jersf_args_jagged = {k: jagged_out[jer_name_map[k]] for k in self.jec_stack.jersf.signature}
                    try:
                        jersf_flat = self.jec_stack.jersf.getScaleFactor(**jersf_args_jagged)
                    except Exception:
                        raise flat_exc
                jet_energy_resolution_scale_factor = _as_jagged_per_jet(jersf_flat, counts, "JER scale factor")

            elif self.tool == "clib":
                jet_energy_resolution = None
                jet_energy_resolution_scale_factor = None
                for jer_entry in self.jec_stack.jer_names_clib:
                    outtag = "jet_energy_resolution"
                    jer_entry = jer_entry.replace("SF", "ScaleFactor")
                    sf = self.corrections[jer_entry]
                    inputs = get_corr_inputs(jets=jagged_out, corr_obj=sf, name_map=jer_name_map)
                    if "ScaleFactor" in jer_entry:
                        outtag += "_scale_factor"
                        nom = _as_jagged_per_jet(ak.values_astype(sf.evaluate(*inputs, "nom"), numpy.float32), counts, outtag)
                        up = _as_jagged_per_jet(ak.values_astype(sf.evaluate(*inputs, "up"), numpy.float32), counts, outtag)
                        down = _as_jagged_per_jet(
                            ak.values_astype(sf.evaluate(*inputs, "down"), numpy.float32), counts, outtag
                        )
                        stacked = ak.concatenate([nom[..., None], up[..., None], down[..., None]], axis=-1)
                        jet_energy_resolution_scale_factor = stacked
                    else:
                        correction = ak.values_astype(sf.evaluate(*inputs), numpy.float32)
                        jet_energy_resolution = _as_jagged_per_jet(correction, counts, outtag)

                if jet_energy_resolution is None:
                    jet_energy_resolution = ak.zeros_like(jagged_out[jer_name_map["JetPt"]])
                if jet_energy_resolution_scale_factor is None:
                    jet_energy_resolution_scale_factor = ak.ones_like(jagged_out[jer_name_map["JetPt"]])[..., None]

            rng = numpy.random.default_rng()
            jet_resolution_rand_gauss = _random_gauss(counts, rng)

            # --- nominal JER smearing (always applied if has_jer) ---
            jer_correction = jer_smear(
                variation=0,
                forceStochastic=self.forceStochastic,
                pt_gen=ak.values_astype(jagged_out[jer_name_map.get("ptGenJet", "ptGenJet")], numpy.float32),
                jetPt=ak.values_astype(jagged_out[jer_name_map["JetPt"]], numpy.float32),
                etaJet=ak.values_astype(jagged_out[jer_name_map["JetEta"]], numpy.float32),
                jet_energy_resolution=ak.values_astype(jet_energy_resolution, numpy.float32),
                jet_resolution_rand_gauss=ak.values_astype(jet_resolution_rand_gauss, numpy.float32),
                jet_energy_resolution_scale_factor=ak.values_astype(jet_energy_resolution_scale_factor, numpy.float32),
            )

            jagged_out["jet_energy_resolution"] = jet_energy_resolution
            jagged_out["jet_energy_resolution_scale_factor"] = jet_energy_resolution_scale_factor
            jagged_out["jet_resolution_rand_gauss"] = jet_resolution_rand_gauss
            jagged_out["jet_energy_resolution_correction"] = jer_correction

            jagged_out[self.name_map["JetPt"]] = jer_correction * jagged_out[jer_name_map["JetPt"]]
            jagged_out[self.name_map["JetMass"]] = jer_correction * jagged_out[jer_name_map["JetMass"]]

            jagged_out[self.name_map["JetPt"] + "_jer"] = jagged_out[self.name_map["JetPt"]]
            jagged_out[self.name_map["JetMass"] + "_jer"] = jagged_out[self.name_map["JetMass"]]

            # --- only build JER up/down systematics if requested ---
            if jer_systematics_enabled:
                def build_jer_variant(variation_index):
                    correction = jer_smear(
                        variation=variation_index,
                        forceStochastic=self.forceStochastic,
                        pt_gen=ak.values_astype(jagged_out[jer_name_map.get("ptGenJet", "ptGenJet")], numpy.float32),
                        jetPt=ak.values_astype(jagged_out[jer_name_map["JetPt"]], numpy.float32),
                        etaJet=ak.values_astype(jagged_out[jer_name_map["JetEta"]], numpy.float32),
                        jet_energy_resolution=ak.values_astype(jet_energy_resolution, numpy.float32),
                        jet_resolution_rand_gauss=ak.values_astype(jet_resolution_rand_gauss, numpy.float32),
                        jet_energy_resolution_scale_factor=ak.values_astype(
                            jet_energy_resolution_scale_factor, numpy.float32
                        ),
                    )

                    var_dict = {field: jagged_out[field] for field in in_dict}
                    var_dict[self.name_map["JetPt"]] = correction * jagged_out[jer_name_map["JetPt"]]
                    var_dict[self.name_map["JetMass"]] = correction * jagged_out[jer_name_map["JetMass"]]
                    return ak.zip(var_dict, depth_limit=1, parameters=parameters, behavior=behavior)

                jer_systematic = ak.zip(
                    {"up": build_jer_variant(1), "down": build_jer_variant(2)},
                    depth_limit=1,
                    with_name="JetSystematic",
                )

        # --- JES (JEC uncertainties) ---
        has_junc = self.jec_stack.junc is not None
        if self.tool == "clib":
            has_junc = len(self.jec_stack.jec_uncsources_clib) > 0

        jes_entries = []
        jes_enabled = has_junc and self._variation_control.allow_jes

        if jes_enabled:
            junc_name_map = dict(self.name_map)

            # IMPORTANT: follow legacy behavior:
            # If JER is configured at all, JES is defined on top of the JER-smeared jets,
            # regardless of whether we built JER up/down variations.
            if has_jer:
                junc_name_map["JetPt"] = junc_name_map["JetPt"] + "_jer"
                junc_name_map["JetMass"] = junc_name_map["JetMass"] + "_jer"
            else:
                junc_name_map["JetPt"] = junc_name_map["JetPt"] + "_jec"
                junc_name_map["JetMass"] = junc_name_map["JetMass"] + "_jec"

            if self.tool == "jecstack":
                junc_args = {k: ak.flatten(jagged_out[junc_name_map[k]]) for k in self.jec_stack.junc.signature}
                raw_juncs = self.jec_stack.junc.getUncertainty(**junc_args)
                jes_entries = [
                    (name, func)
                    for name, func in raw_juncs
                    if self._variation_control.allows_jes_component(name)
                ]

            elif self.tool == "clib":
                for junc_name in self.jec_stack.jec_uncsources_clib:
                    sf = self.corrections[junc_name]
                    if sf is None:
                        raise ValueError(f"Correction {junc_name} not found in self.corrections")

                    inputs = get_corr_inputs(jets=jagged_out, corr_obj=sf, name_map=junc_name_map)
                    # 'unc' here is the relative uncertainty δ; we build (1±δ) factors
                    unc = ak.values_astype(sf.evaluate(*inputs), numpy.float32)
                    jagged_unc = _as_jagged_per_jet(unc, counts, f"JES uncertainty {junc_name}")
                    central = ak.ones_like(jagged_out[self.name_map["JetPt"]])
                    unc_up = central + jagged_unc
                    unc_down = central - jagged_unc

                    name_tokens = junc_name.split("_")
                    component_name = name_tokens[-2] if len(name_tokens) >= 3 else name_tokens[-1]
                    if not self._variation_control.allows_jes_component(component_name):
                        continue

                    jes_entries.append(
                        (
                            component_name,
                            ak.concatenate([unc_up[..., None], unc_down[..., None]], axis=-1),
                        )
                    )

                    
            def build_variation(unc, jetpt, jetpt_orig, jetmass, jetmass_orig, updown):
                factor = ak.flatten(unc[..., updown:updown+1], axis=-1)
                shifted_pt = factor * jetpt_orig
                shifted_mass = factor * jetmass_orig
                
                return shifted_pt, shifted_mass 

            def build_variant(unc, jetpt, jetpt_orig, jetmass, jetmass_orig):
                up_pt, up_mass = build_variation(unc, jetpt, jetpt_orig, jetmass, jetmass_orig, 0)
                down_pt, down_mass = build_variation(unc, jetpt, jetpt_orig, jetmass, jetmass_orig, 1)
                
                up_rec = ak.zip({"pt": up_pt, "mass": up_mass}, with_name="Jet", depth_limit=1)
                dn_rec = ak.zip({"pt": down_pt, "mass": down_mass}, with_name="Jet", depth_limit=1)
                
                return ak.zip(
                    {"up": up_rec, "down": dn_rec}, 
                    depth_limit=1, 
                    with_name="JetSystematic")

            template_pt = jagged_out[junc_name_map["JetPt"]]
            template_mass = jagged_out[junc_name_map["JetMass"]]

            for name, func in jes_entries:
                jagged_unc = _as_jagged_per_jet(func, counts, f"JES uncertainty {name}")
                jagged_out[f"jet_energy_uncertainty_{name}"] = jagged_unc
                jes_systematics[name] = build_variant(
                    jagged_unc,
                    self.name_map["JetPt"],
                    template_pt,
                    self.name_map["JetMass"],
                    template_mass,
                )
                

        # Start with an empty record that has the correct length and structure
        # before making this change, the area field was throwing errors and/or the structure of jets_record did not match events.Jet
        jets_record = ak.zip(
            {"pt": jagged_out[self.name_map["JetPt"]]}, 
            depth_limit=None, 
            behavior=behavior
        )

        # Add every field one-by-one
        # This uses 'ak.with_field', which is more permissive than 'ak.zip' because it doesn't force a global re-broadcast of the whole dictionary.
        for k, v in jagged_out.items():
            if not (k == "JER" or k.startswith("JES_") or k == self.name_map["JetPt"]):
                try:
                    jets_record = ak.with_field(jets_record, v, k)
                except Exception as e:
                    logger.warning(f"Field {k} failed to add: {e}")

        # Restore behavior and parameters
        jets_record = ak.with_parameter(jets_record, "corrected", True)
        jets_record = ak.with_parameter(jets_record, "typename", "Jet")

        jet_counts = ak.num(jets_record)
        if jer_systematic is not None:

            up_pt = ak.unflatten(ak.flatten(jer_systematic.up['pt']), jet_counts)
            up_mass = ak.unflatten(ak.flatten(jer_systematic.up['mass']), jet_counts)
            down_pt = ak.unflatten(ak.flatten(jer_systematic.down['pt']), jet_counts)
            down_mass = ak.unflatten(ak.flatten(jer_systematic.down['mass']), jet_counts)
            
            slim_jer = ak.zip({
                "up": ak.zip({"pt": up_pt, "mass": up_mass}, with_name="Jet"),
                "down": ak.zip({"pt": down_pt, "mass": down_mass}, with_name="Jet")
            }, with_name="JetSystematic")
            
            jets_record = ak.with_field(jets_record, slim_jer, "JER")

        for name, variation in jes_systematics.items():
            up_pt = ak.unflatten(ak.flatten(variation.up['pt']), jet_counts)
            up_mass = ak.unflatten(ak.flatten(variation.up['mass']), jet_counts)
            down_pt = ak.unflatten(ak.flatten(variation.down['pt']), jet_counts)
            down_mass = ak.unflatten(ak.flatten(variation.down['mass']), jet_counts)
            
            slim_jes = ak.zip({
                "up": ak.zip({"pt": up_pt, "mass": up_mass}, with_name="Jet"),
                "down": ak.zip({"pt": down_pt, "mass": down_mass}, with_name="Jet")
            }, with_name="JetSystematic")
            
            jets_record = ak.with_field(jets_record, slim_jes, f"JES_{name}")

        return jets_record

In [133]:
def ApplyJetCorrections(year, corr_type, isData, era, useclib=True, savelevels=False):

    clib_year_map = {
        "2016APV": "2016preVFP_UL",
        "2016preVFP": "2016preVFP_UL",
        "2016": "2016postVFP_UL",
        "2017": "2017_UL",
        "2018": "2018_UL",
        "2022": "2022_Summer22",
        "2022EE": "2022_Summer22EE",
        "2023": "2023_Summer23",
        "2023BPix": "2023_Summer23BPix",
    }
    
    usejecstack = not useclib

    if year not in clib_year_map.keys():
        raise Exception(f"Error: Unknown year \"{year}\".")

    jec_year = clib_year_map[year]

    jet_algo, jec_tag, jec_levels, jer_tag, junc_types = get_jerc_keys(year, isData, era)
    json_path = ttbarEFT_path(f"data/POG/JME/{jec_year}/jet_jerc.json.gz")

    # Create JECStack for clib scenario
    jec_stack = JECStack(
        jec_tag=jec_tag,
        jec_levels=[],
        jer_tag=jer_tag,
        jet_algo=jet_algo,
        junc_types=junc_types,
        json_path=json_path,
        use_clib=useclib,
        savecorr=savelevels
    )

    # Name map for jet or MET corrections
    name_map = {
        'JetPt':    'pt',
        'JetMass':  'mass',
        'JetEta':   'eta',
        'JetPhi':   'phi',
        'JetA':     'area',
        'ptGenJet': 'pt_gen',
        'ptRaw':    'pt',
        'massRaw':  'mass',
        'Rho':      'Rho',
        'METpt':    'pt',
        'METphi':   'phi',
        'UnClusteredEnergyDeltaX': 'MetUnclustEnUpDeltaX',
        'UnClusteredEnergyDeltaY': 'MetUnclustEnUpDeltaY'
    }

    # Return appropriate factory based on correction type
    if corr_type == 'met':
        return CorrectedMETFactory.CorrectedMETFactory(name_map)
    elif corr_type == 'jets':
        return CorrectedJetsFactory(name_map, jec_stack, allowed_variations=None)
    else: 
        raise ValueError(f"Unknown correction type \"{corr_type}\".")

In [138]:
def ApplyJetSystematics(year,cleanedJets,syst_var):

    # getattr(cleanedJets, f"JER.up.pt")
    if (syst_var == 'nominal'):
        return cleanedJets

    if 'Up' in syst_var: 
        suffix = 'up'
        if syst_var.startswith(f'JER_{year}'):
            base_field = "JER"
        elif syst_var == "JESUp":
            base_field = "JES_jes"
        else: 
            base_field = syst_var[:-2]
            
    elif 'Down' in syst_var:
            suffix = 'down'
            if syst_var.startswith(f'JER_{year}'):
                base_field = "JER"
            elif syst_var == "JESDown":
                base_field = "JES_jes"
            else:
                base_field = syst_var[:-4]
    else: 
        raise ValueError(f"Unknown variation {syst_var}.")
        
    if base_field not in cleanedJets.fields:
        raise ValueError(f"Field {base_field} not found in jets for variation {syst_var}")
    
    var_record = getattr(cleanedJets[base_field], suffix)
    new_jets = ak.with_field(cleanedJets, var_record["pt"], "pt")
    new_jets = ak.with_field(new_jets, var_record["mass"], "mass")
    
    return new_jets

In [139]:
correctedJets = ApplyJetCorrections(year, corr_type='jets', isData=isData, era=None).build(cleanedJets)

In [141]:
correctedJets.pt[0]

<Array [182, 73.4, 39.3, 33.8, 20.8, 20.1] type='6 * float32'>

In [144]:
JERup_jets = ApplyJetSystematics(year=year, cleanedJets=correctedJets, syst_var='JER_2017Down')

In [145]:
JERup_jets.pt[0]

<Array [181, 74.3, 39.6, 32.9, 22, 20] type='6 * float32'>

In [136]:
correctedJets.JER.up[0]

<JetArray [Jet, ..., Jet] type='6 * Jet[pt: float32, mass: float32]'>

In [131]:
correctedJets.JER.up[0]

<JetArray [Jet, ..., Jet] type='6 * Jet[pt: float32, mass: float32]'>

In [126]:
correctedJets.JER.up[0]

<JetArray [Jet, ..., Jet] type='6 * Jet[pt: float32, mass: float32]'>

In [117]:
cleanedJets[0].pt

<Array [182, 75.4, 40, 33.1, 22, 19.3] type='6 * float32[parameters={"__doc...'>

In [118]:
cleanedJets[0].mass

<Array [17.8, 16.2, 8.04, ..., 4.94, 3.61] type='6 * float32[parameters={"_...'>

In [62]:
ak.flatten(correctedJets.JER.up[0].pt, axis=-1)

<Array [183, 72.5, 39, 34.6, ..., 39, 34.6, 26.4, 22.6] type='36 * float32'>